In [1]:
import chromadb , json, ollama

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 341 docs


In [2]:
PATH = "data/lufthansa_labeled.json"
labeled = json.load(open(PATH, encoding="utf-8"))

for d in labeled:
    if d["category"] == "opportunity":
        r = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content":
            "Rate the business impact of this opportunity for Lufthansa. "
            "Answer with exactly ONE word — High, Medium, or Low.\n\n" + d["text"]}])
        ans = r["message"]["content"].strip().split()[0].strip(".,").capitalize()
        d["impact"] = ans if ans in ("High", "Medium", "Low") else "Medium"
        print(d["text"][:55], "->", d["impact"])

json.dump(labeled, open(PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved impact for", sum(d["category"] == "opportunity" for d in labeled), "opportunities")

Lufthansa Group Orders More A350s and B787s - AeroMorni -> Medium
Lufthansa Alternatives & Competitors - SaaSHub. Find th -> Low
lufthansa.com Traffic Analytics, Ranking & Audience. |  -> Medium
Lufthansa: Revenue, Competitors, Alternatives. Lufthans -> Low
Lufthansa Competitors and Alternatives - Owler. Lufthan -> Medium
r/AskGermany on Reddit: Is it true that “Lufthansa” is  -> Low
Award Travel Tools : awardtravel. They are part of Boar -> Medium
AwardSense: Find award flights from the most popular ai -> Medium
Lufthansa flight delay compensation review : r/Flights. -> Medium
EPA:AIRF Financials | Air France KLM SA - Investing.com -> Low
KLM Royal Dutch Airlines - Book flights online - KLM US -> Medium
KLM celebrates 90 years of air travel to Stockholm - TT -> Medium
Online check-in Air France | Air France, France. AIR FR -> Medium
Pointe à Pitre – Atlanta: a new Air France route - Avia -> Low
Ryanair SWOT Analysis (2025). Ryanair Opportunities. Ex -> Medium
easyJet: Hidden Asset Val

In [3]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

In [4]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("data/recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

 Saved 5 recommendations


CEO Briefing (Section 7)

In [7]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r.get('recommendation','')} (priority {r.get('priority','?')}, risk {r.get('risk_level','?')})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these 3 keys:
- "what_happened": a single plain-text string (2-3 sentences)
- "why_it_matters": a single plain-text string (2-3 sentences)
- "what_to_do_next": a single plain-text string (2-3 sentences)
Each value MUST be a plain string — NOT a nested object, dict, or list.
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [8]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("data/ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": "Recent market analysis suggests we must prioritize digitalization and customer satisfaction to stay competitive.",
  "why_it_matters": "Investing in B2B travel experiences and improving customer satisfaction are critical to boosting loyalty and competitiveness, while aggressive pricing strategies will help us adapt to changing market conditions.",
  "what_to_do_next": "We need to accelerate the implementation of structural transformation, invest in digital services, and launch an ambitious plan to improve customer satisfaction and loyalty through targeted initiatives."
}


### REWORK

In [9]:
from retrieval import semantic_search, bm25_search, hybrid_search

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
def retrieve_evidence(query, k_each=5, final_k=5):
    """Run all 3 retrievers, pool results, dedup by URL, keep the best by consensus."""
    methods = {
        "semantic": semantic_search(query, k_each),
        "bm25":     bm25_search(query, k_each),
        "hybrid":   hybrid_search(query, k_each),
    }

    seen = {}                                   # url -> {doc, hits, rank_sum}
    for docs_list in methods.values():
        for rank, d in enumerate(docs_list):    # rank 0 = top of that method
            key = d["url"]
            if key not in seen:
                seen[key] = {"doc": d, "hits": 0, "rank_sum": 0}
            seen[key]["hits"]     += 1           # how many methods found it
            seen[key]["rank_sum"] += rank        # how high they ranked it

    # best = found by MOST methods, tie-break by best average rank
    ranked = sorted(seen.values(), key=lambda x: (-x["hits"], x["rank_sum"]))
    return [x["doc"] for x in ranked[:final_k]]

In [11]:
import ollama, json

def make_plan(goal):
    """Break the CEO's abstract goal into specific, searchable sub-questions."""
    system = (
        "You are a research planner for a strategic intelligence agent about Lufthansa. "
        "Break the user's question into 2-4 SPECIFIC, keyword-rich sub-questions "
        "that will retrieve good evidence from a news database. "
        'Return JSON exactly like: {"steps": ["...", "...", "..."]}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": goal},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["steps"]

In [12]:
def gather_evidence(goal, final_per_step=4):
    """Plan the goal, retrieve for each sub-question, pool + dedup the evidence."""
    plan = make_plan(goal)                              # 1. break goal into sub-questions
    print(f"🧭 PLANNING:{goal}")
    for s in plan:
        print("   -", s)

    pooled = []
    for sub_q in plan:                                  # 2. retrieve for EACH sub-question
        docs = retrieve_evidence(sub_q, final_k=final_per_step)
        print(f"   🔎 {sub_q[:45]}... → {len(docs)} docs")
        pooled.extend(docs)                             # add them all to one big list

    unique = list({d["url"]: d for d in pooled}.values())  # 3. dedup by URL
    print(f"\n📊 Pooled {len(pooled)} → {len(unique)} unique evidence docs")
    return unique

In [13]:
from retrieval import docs as labeled_docs   # the full labeled docs (have category/sentiment/severity)
from collections import Counter

label_by_url = {d["url"]: d for d in labeled_docs}   # URL → full labeled doc (lookup table)
#print (label_by_url)
def analyze(evidence):
    """Attach each doc's Task-4 label (by URL) and summarize what we found."""
    for d in evidence:
        full = label_by_url.get(d["url"], {})         # find the labeled version by URL
        d["category"]  = full.get("category", "unknown") # ('key',default if not found)
        d["sentiment"] = full.get("sentiment", "unknown")
        d["severity"]  = full.get("severity")         # only risks have this

    counts = Counter(d["category"] for d in evidence)  # how many of each category
    print("📊 ANALYSIS:")
    for cat, n in counts.items():
        print(f"   {cat}: {n}")
    return evidence, counts

In [14]:
def decide_enough(goal, evidence, counts):
    """Ask the LLM whether the evidence found is enough to answer the goal."""
    summary = ", ".join(f"{n} {cat}" for cat, n in counts.items())   # "9 trend, 6 risk"
    sample  = " | ".join(d["text"][:70] for d in evidence[:4])       # peek at a few docs

    system_prompt = (                          
        "You are the decision step of a research agent about Lufthansa. "
        "Decide if there is AT LEAST SOME relevant evidence to give a reasonable answer. "
        "You do NOT need perfect or exhaustive coverage — if several documents relate to the "
        "question, that is ENOUGH. Only answer false if the evidence is clearly off-topic or nearly empty. "
        'Return JSON: {"sufficient": true or false, "reason": "one short sentence"}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Question: {goal}\nEvidence: {summary}\nSamples: {sample}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])

In [15]:
def reformulate(goal, reason):
    """Rewrite the question to be more specific when the evidence was insufficient."""
    system = (
        "The previous search for this question did not find specific enough evidence. "
        "Rewrite it into ONE more specific, focused question that targets the missing information. "
        'Return JSON: {"new_goal": "..."}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": f"Question: {goal}\nWhy it failed: {reason}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["new_goal"]

In [16]:
def recommend(goal, evidence):
    """Generate the structured recommendation from the analyzed evidence."""
    context = "\n\n".join(
        f"[{d['source']} · {d.get('category','?')}] {d['text']}" for d in evidence
    )

    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Evidence:\n{context}\n\nQuestion: {goal}"},
        ],
        format="json",
    )
    rec = json.loads(res["message"]["content"])
    rec["question"] = goal
    rec["sources"]  = [d["url"] for d in evidence]
    return rec

### VALIDATE 

In [17]:
def validate(rec, evidence):
    """Check the recommendation is grounded in the evidence (no invented facts)."""
    ev = " | ".join(d["text"][:500] for d in evidence)
    system = (
        "You are the validation step of a research agent about Lufthansa. "
        "Check whether the recommendation and its justification are SUPPORTED by the evidence. "
        "If any claim is not backed by the evidence, it is invalid. "
        'Return JSON: {"valid": true or false, "reason": "one short sentence"}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content":
                f"Recommendation: {rec['recommendation']}\n"
                f"Justification: {rec['justification']}\n"
                f"Evidence: {ev}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])

### conversation memory

In [18]:
def resolve_goal(goal, history):
    """Use conversation history to rewrite a follow-up into a standalone question."""
    if not history:
        return goal
    convo = "\n".join(f"Q: {t['question']}\nA: {t['recommendation']}" for t in history)
    system = (
        "Rewrite the user's latest question into a STANDALONE question that includes any context "
        "it refers to from the conversation (resolve words like 'that', 'it', 'this'). "
        "If it is already standalone, return it unchanged. "
        'Return JSON: {"goal": "..."}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": f"Conversation so far:\n{convo}\n\nLatest question: {goal}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["goal"]

### THE ORCHESTRATOR

In [19]:
def run_agent(goal,history=None, max_tries=2):
    goal = resolve_goal(goal, history)
    print(f"\n🎯 GOAL: {goal}\n")

    # round 1: gather → analyze → decide
    evidence = gather_evidence(goal)  # plan + retrieve
    analyzed, counts = analyze(evidence) # sort by labels
    verdict = decide_enough(goal, analyzed, counts) # enough?
    print("🤔 DECIDE:", verdict["sufficient"], "—", verdict["reason"])

    # self-correction loop: capped, ACCUMULATING evidence (your idea + the drift fix)
    tries = 0
    while not verdict["sufficient"] and tries < max_tries:
        print(f"\n🔄 Not enough — reformulating (try {tries+1}/{max_tries})")
        sharper = reformulate(goal, verdict["reason"])
        evidence += gather_evidence(sharper)                          # ADD, don't replace
        evidence = list({d["url"]: d for d in evidence}.values())     # dedup
        analyzed, counts = analyze(evidence)
        verdict = decide_enough(goal, analyzed, counts)               # judge vs ORIGINAL goal
        print("🤔 DECIDE:", verdict["sufficient"], "—", verdict["reason"])
        tries += 1

    # ← loop exited because SUFFICIENT or CAP hit → recommend either way (your logic!)
    print(f"\n📝 RECOMMENDING on {len(evidence)} docs...")
    rec = recommend(goal, analyzed)

    # validate (one redo if not grounded)
    check = validate(rec, analyzed)
    print("🛡️ VALIDATE:", check)
    if not check["valid"]:
        print("   ↻ regenerating...")
        rec = recommend(goal, analyzed)

    rec["evidence_sufficient"] = verdict["sufficient"]   # honest flag for the dashboard
    return rec

In [20]:
final = run_agent("What are the biggest risks for Lufthansa?")
import json
print(json.dumps(final, indent=2, ensure_ascii=False))


🎯 GOAL: What are the biggest risks for Lufthansa?

🧭 PLANNING:What are the biggest risks for Lufthansa?
   - Lufthansa's market share and revenue decline due to increased competition from low-cost carriers
   - Effectiveness of Lufthansa's response to COVID-19 pandemic on passenger demand and airline operations
   - Financial impact of grounding its fleet in 2020 and subsequent losses
   - Sustainability goals and carbon offsetting strategies for reducing environmental impact
   🔎 Lufthansa's market share and revenue decline ... → 4 docs
   🔎 Effectiveness of Lufthansa's response to COVI... → 4 docs
   🔎 Financial impact of grounding its fleet in 20... → 4 docs
   🔎 Sustainability goals and carbon offsetting st... → 4 docs

📊 Pooled 16 → 16 unique evidence docs
📊 ANALYSIS:
   trend: 7
   risk: 6
   opportunity: 3
🤔 DECIDE: True — Several documents mention financial losses and challenges.

📝 RECOMMENDING on 16 docs...
🛡️ VALIDATE: {'valid': True, 'reason': "The evidence suggests Luftha